# Notebook 05: Multimodal Fusion Model

Combine image features from MobileNetV2 and tabular features (age, gender) using a late-fusion Keras Functional API model.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

sns.set_style("whitegrid")
BASE_DIR = Path("..")
CLASSES = ["Normal", "Mild", "Moderate", "Severe"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE
RANDOM_STATE = 42


## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv(BASE_DIR / "data" / "metadata.csv")

le_label = LabelEncoder(); le_label.fit(CLASSES)
df["label"] = le_label.transform(df["diagnosis"])

le_gender = LabelEncoder()
df["gender_enc"] = le_gender.fit_transform(df["gender"])

scaler = StandardScaler()
df["age_scaled"] = scaler.fit_transform(df[["age"]])

df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df["diagnosis"], random_state=RANDOM_STATE
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["diagnosis"], random_state=RANDOM_STATE
)
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")


## 3. Multimodal tf.data Pipeline

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
], name="augmentation")


def make_multimodal_dataset(dataframe, augment=False, shuffle=True):
    """Return a tf.data.Dataset yielding ((image, tabular), label) tuples.

    Args:
        dataframe: DataFrame with image_path, age_scaled, gender_enc, label.
        augment: Apply random augmentations to images.
        shuffle: Shuffle dataset order.

    Returns:
        Batched, prefetched tf.data.Dataset.
    """
    paths    = [str(BASE_DIR / p) for p in dataframe["image_path"]]
    tabular  = dataframe[["age_scaled", "gender_enc"]].values.astype(np.float32)
    labels   = dataframe["label"].values

    def load_image(path):
        raw   = tf.io.read_file(path)
        image = tf.image.decode_png(raw, channels=3)
        image = tf.image.resize(image, IMG_SIZE)
        return tf.cast(image, tf.float32) / 255.0

    path_ds = tf.data.Dataset.from_tensor_slices(paths).map(
        load_image, num_parallel_calls=AUTOTUNE
    )
    tab_ds  = tf.data.Dataset.from_tensor_slices(tabular)
    lbl_ds  = tf.data.Dataset.from_tensor_slices(labels)

    ds = tf.data.Dataset.zip((path_ds, tab_ds, lbl_ds))
    if shuffle:
        ds = ds.shuffle(len(dataframe), seed=RANDOM_STATE)
    if augment:
        ds = ds.map(
            lambda img, tab, lbl: (augmentation(img, training=True), tab, lbl),
            num_parallel_calls=AUTOTUNE
        )
    ds = ds.map(lambda img, tab, lbl: ((img, tab), lbl))
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_multimodal_dataset(df_train, augment=True)
val_ds   = make_multimodal_dataset(df_val,   augment=False, shuffle=False)
test_ds  = make_multimodal_dataset(df_test,  augment=False, shuffle=False)

for (images, tab), labels in train_ds.take(1):
    print(f"Image batch shape   : {images.shape}")
    print(f"Tabular batch shape : {tab.shape}")
    print(f"Label batch shape   : {labels.shape}")


## 4. Multimodal Fusion Model Architecture

In [ ]:
def build_fusion_model(
    num_classes: int = 4,
    tabular_dim: int = 2,
    img_embedding_dim: int = 64,
    tab_embedding_dim: int = 16,
    dropout_rate: float = 0.4,
) -> Model:
    """Multimodal fusion model combining MobileNetV2 image branch and tabular branch.

    Image branch:
        MobileNetV2 (frozen) → GlobalAveragePooling2D → Dense(img_embedding_dim)

    Tabular branch:
        Dense(tab_embedding_dim, relu) → BatchNormalisation

    Fusion:
        Concatenate → Dense(64, relu) → Dropout → Dense(num_classes, softmax)

    Args:
        num_classes: Output classes (4 severity levels).
        tabular_dim: Number of tabular input features (age, gender = 2).
        img_embedding_dim: Size of image embedding vector.
        tab_embedding_dim: Size of tabular embedding vector.
        dropout_rate: Dropout rate in the fusion head.

    Returns:
        Compiled Keras Model.
    """
    # --- Image branch ---
    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False

    img_input = Input(shape=(*IMG_SIZE, 3), name="image_input")
    x_img = base_model(img_input, training=False)
    x_img = layers.GlobalAveragePooling2D(name="img_gap")(x_img)
    x_img = layers.Dense(img_embedding_dim, activation="relu", name="img_embedding")(x_img)

    # --- Tabular branch ---
    tab_input = Input(shape=(tabular_dim,), name="tabular_input")
    x_tab = layers.Dense(tab_embedding_dim, activation="relu", name="tab_dense")(tab_input)
    x_tab = layers.BatchNormalization(name="tab_bn")(x_tab)

    # --- Fusion head ---
    fused = layers.Concatenate(name="fusion")([x_img, x_tab])
    fused = layers.Dense(64, activation="relu", name="fusion_dense")(fused)
    fused = layers.Dropout(dropout_rate, name="fusion_dropout")(fused)
    output = layers.Dense(num_classes, activation="softmax", name="predictions")(fused)

    model = Model(
        inputs=[img_input, tab_input],
        outputs=output,
        name="MultimodalFusion"
    )
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


fusion_model = build_fusion_model()
fusion_model.summary()


## 5. Class Weights

In [ ]:
y_train = df_train["label"].values
cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(cw))
print("Class weights:", {CLASSES[k]: round(v, 3) for k, v in class_weight_dict.items()})


## 6. Train Fusion Model

In [ ]:
os.makedirs("../models/saved_models", exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ModelCheckpoint(
        "../models/saved_models/fusion_model_best.h5",
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
]

history = fusion_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history.history["accuracy"]) + 1)

axes[0].plot(epochs, history.history["accuracy"],     label="Train")
axes[0].plot(epochs, history.history["val_accuracy"], label="Val")
axes[0].set_title("Fusion Model — Accuracy"); axes[0].legend()

axes[1].plot(epochs, history.history["loss"],     label="Train")
axes[1].plot(epochs, history.history["val_loss"], label="Val")
axes[1].set_title("Fusion Model — Loss"); axes[1].legend()

plt.tight_layout()
plt.savefig("../models/saved_models/fusion_training_history.png", bbox_inches="tight")
plt.show()


## 8. Evaluate on Test Set

In [ ]:
y_true, y_pred_all = [], []
for (images, tab_feats), labels in test_ds:
    preds = fusion_model.predict([images, tab_feats], verbose=0)
    y_true.extend(labels.numpy())
    y_pred_all.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred_all)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"Test Accuracy : {acc:.4f}")
print(f"Test Macro F1 : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(4)))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Multimodal Fusion — Confusion Matrix (Test)", fontweight="bold")
plt.tight_layout()
plt.savefig("../models/saved_models/fusion_confusion_matrix.png", bbox_inches="tight")
plt.show()

severe_idx = le_label.transform(["Severe"])[0]
mask = y_true == severe_idx
if mask.sum() > 0:
    severe_recall = (y_pred[mask] == severe_idx).mean()
    print(f"\nSevere Anemia Recall: {severe_recall:.4f}  (critical safety metric)")


## 9. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
all_labels = df["label"].values
all_indices = np.arange(len(df))

cv_acc, cv_f1 = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(all_indices, all_labels), 1):
    fold_train = df.iloc[train_idx].reset_index(drop=True)
    fold_val   = df.iloc[val_idx].reset_index(drop=True)

    fold_train_ds = make_multimodal_dataset(fold_train, augment=True)
    fold_val_ds   = make_multimodal_dataset(fold_val,   augment=False, shuffle=False)

    cw_fold = compute_class_weight("balanced", classes=np.unique(fold_train["label"].values),
                                   y=fold_train["label"].values)
    cw_dict_fold = dict(enumerate(cw_fold))

    fold_model = build_fusion_model()
    fold_model.fit(fold_train_ds, epochs=20, verbose=0, class_weight=cw_dict_fold)

    y_v = fold_val["label"].values
    y_p = []
    for (imgs, tabs), _ in fold_val_ds:
        preds = fold_model.predict([imgs, tabs], verbose=0)
        y_p.extend(np.argmax(preds, axis=1))
    y_p = np.array(y_p)

    cv_acc.append(accuracy_score(y_v, y_p))
    cv_f1.append(f1_score(y_v, y_p, average="macro", zero_division=0))
    print(f"Fold {fold}: Acc={cv_acc[-1]:.4f}  Macro F1={cv_f1[-1]:.4f}")

print(f"\nCV Accuracy : {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"CV Macro F1 : {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
